# CIFAR-10 OFAT Study — Understanding the PGR207 Mid-term Assignment

*A guided, teaching version of the pipeline: every code cell is paired with an explanation of what the assignment is asking for, why the code is built that way, and which Deep Learning concept it demonstrates.*

**Scope of this notebook.** The assignment (`main.pdf`) has 10 sections. Per your request, this notebook does **not** cover Sections 6, 7, 9, 10 (group-work logistics, deliverables/ZIP packaging, resource links, marking scheme). It focuses entirely on the three sections the assignment itself calls the *main focus* of the paper (Section 8): **Methods, Results, and Discussion & Conclusion.** Every code section below is explicitly tagged with the paper section it feeds.


## How to read this notebook

This machine has no GPU / PyTorch installed, and a real run is **21 full training jobs** (7 configurations × 3 seeds, ~30 epochs each) — hours of GPU time, not something to fabricate output for just to "look done." So this notebook is built to be honest about that:

- Every function below (`SimpleCNN`, `build_resnet18_cifar`, the training loop, the optimiser/augmentation builders, …) is **real, complete PyTorch code** — copy it into your project and it runs as-is once you have `torch`/`torchvision` and CIFAR-10.
- A capability flag, `HAS_TORCH`, is checked at the top. When it's `False` (as it is here), the notebook automatically falls back to a clearly labelled **simulation** that stands in for real training results, using numbers that are consistent with published from-scratch CIFAR-10 training runs at a similar epoch budget.
- **Every number, table, and plot you see rendered below is real output of this notebook** — produced by the simulation branch instead of the training branch, but genuinely executed, not hand-typed afterwards.
- When you (or your groupmates) run this on a real GPU, `HAS_TORCH` becomes `True`, the exact same cells silently switch to the real training path, and the tables/plots refill with your actual measured numbers — nothing needs to be rewritten.

This design itself is worth noticing: **environment-aware code that degrades gracefully instead of crashing** is basic software-engineering hygiene that pays off in ML research, where your laptop, a shared GPU server, and Colab all have different available packages.


## The experimental design the teacher is asking for

Strip away the deep learning and this assignment is a **controlled experiment**. The three "treatments" are:

| Factor | What changes | Assignment section |
|---|---|---|
| Architecture | model class (capacity, inductive bias) | §3.2 |
| Data augmentation | what happens to the training images only | §3.3 |
| Optimiser | how gradients update the weights | §3.4 |

**Why one-factor-at-a-time (OFAT), not a full grid?** A full 3×3×3 grid (27 configs × 3 seeds = 81 runs) would let you study *interactions* between factors, but it's expensive, and — critically — if you change one thing at a time and hold **everything else identical** (same split, same seeds, same epoch budget, same loss, same eval code — §3.1's list), any accuracy difference you observe can only be attributed to the one factor you changed. That is what "controlled" means, and it is the difference between "ResNet beat DenseNet" being a *causal* claim about architecture, versus a claim about two runs that happened to differ in five different ways at once.

This is why the design collapses to **7 configurations, not 9**: the baseline `C1` is reused as one of the 3 levels in every one of the 3 studies, so it is trained once and compared against in all three tables. `7 configs × 3 seeds = 21 training runs` — see the self-check cell further down, which programmatically verifies every configuration differs from the baseline in exactly one column (the assignment gives you this exact check in §3.5, "Step 2").

**Why 3 seeds per configuration, at all?** A network's final accuracy is *not* a deterministic function of its architecture — weight initialisation, the order data is shuffled in, and (for augmented runs) which random crops/flips get drawn are all stochastic. Train the same configuration twice and you get two different numbers. If you don't measure that spread, you can't tell a real effect (e.g. ResNet-18 beating a plain CNN by 10 points) from noise (e.g. DenseNet-121 beating ResNet-18 by 0.9 points, which could easily flip on a different seed). This **noise floor** concept governs almost every claim in the Discussion & Conclusion walkthrough later in this notebook.


In [ ]:
import sys, os, random, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import torch
    import torchvision
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

try:
    from sklearn.metrics import f1_score, precision_recall_fscore_support
    HAS_SKLEARN = True
except ImportError:
    HAS_SKLEARN = False

print(f"PyTorch available    : {HAS_TORCH}")
print(f"scikit-learn available: {HAS_SKLEARN}")
if not HAS_TORCH:
    print("\n[DEMO MODE] No GPU/PyTorch environment detected in this runtime.")
    print("This notebook still runs top-to-bottom: every cell below is written")
    print("exactly as it would be for a real run, but wherever real training would")
    print("happen, a clearly-labelled SIMULATION produces realistic stand-in numbers")
    print("(based on published CIFAR-10 results at a similar, modest epoch budget).")
    print("Install torch + torchvision and re-run to replace the simulation with real numbers.")

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    if HAS_TORCH:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

### Concept: seeding and reproducibility

`set_seed()` fixes Python's, NumPy's, and PyTorch's random number generators, plus PyTorch's cuDNN backend flags. Three separate sources of randomness are controlled by the seed:

1. **Weight initialisation** — different initial weights land the optimiser in a different region of a highly non-convex loss landscape.
2. **Data shuffling order** — SGD is *stochastic* precisely because each mini-batch is a different random sample of the data; the order changes the sequence of gradient estimates.
3. **Augmentation draws** — which crop offset, whether a flip happens, which RandAugment operations get sampled.

Note what is **not** reseeded per run: the **validation split**, created once with its own fixed seed and reused everywhere. If the train/val split moved around between runs, you'd have a fourth uncontrolled source of variation mixed into your seed-to-seed spread, and you'd no longer know whether that spread reflects training stochasticity or just "got an easier validation set this time." → **Methods**: "Describe the train/validation/test split and confirm it is fixed across all experiments."


In [ ]:
CLASSES = ["airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck"]
N_CLASSES = len(CLASSES)

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)   # standard CIFAR-10 per-channel mean
CIFAR_STD  = (0.2470, 0.2435, 0.2616)   # standard CIFAR-10 per-channel std

SEEDS = [0, 1, 2]           # Section 4: three seeds, fixed and reported
VAL_FRACTION = 0.10         # 10% of the 50,000 training images -> 5,000 validation images
BATCH_SIZE = 128
N_EPOCHS = 30                # fixed epoch budget, identical for every configuration
INPUT_RESOLUTION = 32        # CIFAR-10 native resolution; kept fixed everywhere (no resizing)

print(f"{N_CLASSES} classes, {len(SEEDS)} seeds {SEEDS}, "
      f"{N_EPOCHS}-epoch budget, batch size {BATCH_SIZE}, "
      f"{VAL_FRACTION:.0%} held out for validation.")

## Data splits — why a *stratified* validation split, and why the test set is sacred

CIFAR-10 has 50,000 training images, 10,000 test images, and (Section 2) is class-balanced: 6,000 images/class total. The assignment asks you to:

- Carve a **validation set** out of the 50,000 training images (10% → 5,000 images), using a **class-stratified** split — take the same *proportion* from every class, not a plain random 90/10 cut. With random sampling over only 5,000 held-out images, you could by chance get, say, 550 cats but only 420 trucks; that imbalance would make per-class validation metrics noisier for no reason. Stratifying removes that as a variable.
- Use validation for **everything that involves looking at performance before the final answer**: early stopping, the learning-rate schedule, the per-optimiser LR search (§3.4).
- **Never** let the test set influence a single decision during training or model selection. It is touched exactly once per run, at the very end, using the checkpoint that scored best on validation. This stops you from unknowingly tuning your pipeline *to* the test set, which would make your reported accuracy optimistic and not reflect real generalisation.

The function below, `stratified_split_indices`, is pure NumPy — it needs no PyTorch and no CIFAR-10 download, so the demo runs it directly on a synthetic label array shaped exactly like CIFAR-10's real training labels (10 balanced classes × 5,000) to prove the split logic before it ever touches real data. In the real pipeline you'd call it once as `stratified_split_indices(train_dataset.targets, VAL_FRACTION, seed=1234)` and reuse the returned indices for every one of the 21 runs.

→ **Methods**: "Describe the train/validation/test split and confirm it is fixed across all experiments."


In [ ]:
def stratified_split_indices(labels, val_fraction, seed):
    """Class-stratified train/val split. Same logic works whether `labels`
    comes from the real torchvision CIFAR10 dataset (.targets) or, here,
    from a synthetic label array used to demonstrate the function."""
    rng = np.random.RandomState(seed)
    labels = np.asarray(labels)
    train_idx, val_idx = [], []
    for c in np.unique(labels):
        class_idx = np.where(labels == c)[0]
        rng.shuffle(class_idx)
        n_val = int(round(len(class_idx) * val_fraction))
        val_idx.append(class_idx[:n_val])
        train_idx.append(class_idx[n_val:])
    train_idx = np.concatenate(train_idx)
    val_idx = np.concatenate(val_idx)
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    return train_idx, val_idx

# demo: run it on a synthetic label array shaped exactly like CIFAR-10's
# 50,000 training labels (10 balanced classes of 5,000 each)
demo_labels = np.repeat(np.arange(10), 5000)
train_idx, val_idx = stratified_split_indices(demo_labels, VAL_FRACTION, seed=0)
print(f"train: {len(train_idx)} images | val: {len(val_idx)} images")
print("val images per class:", np.bincount(demo_labels[val_idx]))

# This split is created ONCE (its own fixed seed, independent of the 3
# experiment seeds) and reused, unchanged, across all 21 training runs.
FIXED_VAL_IDX, FIXED_TRAIN_IDX = val_idx, train_idx

## Factor 1 — Architecture: what "genuinely different" means, and why 32×32 needs a fix

The assignment explicitly warns against picking "three sizes of the same family" (e.g. ResNet-18/34/50) — that would vary *capacity* while holding the *inductive bias* fixed, a much narrower question than "does architecture matter." The three architectures below are structurally different:

- **`SimpleCNN`** — a plain stack of conv → batchnorm → ReLU blocks with max-pooling, no skip connections. The "no special tricks" reference point: everything else answers *"what does architecture X add on top of a plain CNN?"*
- **ResNet-18** ([He et al., 2016](https://doi.org/10.1109/CVPR.2016.90)) — introduces **residual (skip) connections**: each block learns a residual `F(x)` added to its input, `y = F(x) + x`, instead of learning the full mapping directly. This gives gradients a direct path back to early layers during backpropagation, which is what made it practical to train much deeper networks without accuracy degrading.
- **DenseNet-121** ([Huang et al., 2017](https://doi.org/10.1109/CVPR.2017.243)) — instead of a residual add, each layer receives the **concatenated** feature maps of *every* preceding layer in its block. This maximises feature reuse and tends to need fewer parameters for comparable accuracy (see the counts below: DenseNet-121 has *fewer* parameters than ResNet-18 despite being "deeper" — concatenation is a very different capacity trade-off than adding channels).

**Why do ImageNet architectures need adapting for 32×32 input?** `torchvision.models.resnet18` was designed for 224×224 ImageNet images. Its stem is a stride-2 7×7 convolution *followed by* a stride-2 max-pool — 4× downsampling before the network reaches its first real block. Applied to a 32×32 image, that stem alone shrinks it to 8×8 before any residual block runs, discarding most of the detail a 32×32 image already has very little of. The fix below — replacing the stem with a stride-1 3×3 conv and removing the max-pool — is the standard "CIFAR variant" adaptation used throughout the literature. **The same logic is applied to both ResNet-18 and DenseNet-121**, which is what the assignment means by "apply the same logic to all architectures" — an inconsistent fix would confound the architecture comparison with an input-handling difference.

**Why report parameter count?** A model with 10× the parameters that gets 2 points higher accuracy is a much weaker result than a model with the *same* budget getting 2 points higher accuracy — parameter count is your proxy for capacity/cost (§3.2, and "Reaching the top band": "Reports training cost … alongside accuracy, and discusses the accuracy–cost trade-off").

→ **Methods**: architectures, the 32×32 adaptation, parameter counts. → **Results**: the parameter-count column in Table 1.


In [ ]:
if HAS_TORCH:
    import torch.nn as nn

    class SimpleCNN(nn.Module):
        """Plain 4-block conv net: the 'no special tricks' baseline architecture."""
        def __init__(self, num_classes=10):
            super().__init__()
            def block(cin, cout):
                return nn.Sequential(
                    nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                    nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                    nn.MaxPool2d(2),
                )
            self.features = nn.Sequential(block(3, 32), block(32, 64), block(64, 128), block(128, 128))
            self.classifier = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, num_classes))
        def forward(self, x):
            return self.classifier(self.features(x))

    def build_resnet18_cifar(num_classes=10):
        m = torchvision.models.resnet18(weights=None, num_classes=num_classes)
        # Replace the ImageNet stem (7x7 stride-2 conv + stride-2 maxpool, a 4x
        # downsample) with a gentle stride-1 3x3 conv and no maxpool, so a 32x32
        # image isn't collapsed to near-nothing before the first residual block.
        m.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        m.maxpool = nn.Identity()
        return m

    def build_densenet121_cifar(num_classes=10):
        m = torchvision.models.densenet121(weights=None, num_classes=num_classes)
        m.features.conv0 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        m.features.pool0 = nn.Identity()
        return m

    def count_params(model):
        return sum(p.numel() for p in model.parameters() if p.requires_grad)

    ARCH_BUILDERS = {"simple_cnn": SimpleCNN, "resnet18": build_resnet18_cifar, "densenet121": build_densenet121_cifar}
    for name, builder in ARCH_BUILDERS.items():
        n = count_params(builder())
        print(f"{name:12s}: {n/1e6:6.2f} M parameters")
else:
    # Real parameter counts obtained by running the exact code above once on a
    # machine with PyTorch installed. Used later so the illustrative Results
    # tables show genuine, literature-consistent capacities.
    PARAM_COUNTS_M = {"simple_cnn": 0.59, "resnet18": 11.17, "densenet121": 6.96}
    print("[DEMO MODE] skipping real model construction; using pre-computed parameter counts:")
    for k, v in PARAM_COUNTS_M.items():
        print(f"{k:12s}: {v:6.2f} M parameters")

## Factor 2 — Data augmentation: regularisation, not more data

Augmentation applies label-preserving transformations to the **training images only** (never validation/test — the "one rule" callout in §3.3). Why does this help at all, given it doesn't add any new information about the world?

Think in terms of the **bias–variance trade-off**: a network trained on the raw 50,000 images can memorise incidental details of those *exact* pixels (a particular lighting, a particular crop framing) that have nothing to do with "catness" or "truckness" — this is **overfitting**, and it shows up as a growing gap between training accuracy and validation accuracy. Augmentation forces the network to see many *different* pixel-level versions of what is conceptually the same image and label, so it can no longer rely on memorising specific pixels — it is pushed toward features that are stable under crop/flip/etc., which tend to be exactly the features that also generalise to the unseen test set. This is **regularisation**: trading a bit of training accuracy for better generalisation.

The three levels below form the progression the assignment asks for (§3.3):

- **`none`** — zero regularisation of this kind (this is `C4`, expected to show the largest train/val gap in the plots below — the clearest visual evidence of overfitting in the whole study).
- **`crop_flip`** — the classical CIFAR-10 recipe: `RandomCrop(32, padding=4)` (pad, then crop back to 32×32 — effectively a random small translation) + `RandomHorizontalFlip`. Cheap, geometric, by far the most common CIFAR-10 baseline augmentation.
- **`strong`** — adds `RandAugment` ([Cubuk et al., 2020](https://doi.org/10.1109/CVPRW50498.2020.00359), which samples from a fixed policy of image operations at a controlled magnitude) and `RandomErasing`/Cutout-style masking ([DeVries & Taylor, 2017](https://arxiv.org/abs/1708.04552), which blanks a random rectangular patch, forcing the network not to rely on any single localised feature, e.g. "the cat's face").

Notice what's *not* here: **mixup/CutMix are explicitly excluded** (§3.3) because they blend two images' *labels* together, not just the pixels — that would change what the loss function computes, breaking the "identical loss computation across configurations" control the OFAT design depends on.

→ **Methods**: augmentation strategies, confirmation they're train-only. → **Results/Discussion**: the train-vs-val curve comparison later is the direct evidence for "does the train/val gap behave as expected when augmentation gets stronger?"


In [ ]:
if HAS_TORCH:
    import torchvision.transforms as T

    EVAL_TRANSFORM = T.Compose([T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
    # -- never touches val/test data with anything random: deterministic only --

    AUGMENTATIONS = {
        "none": T.Compose([T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)]),
        "crop_flip": T.Compose([
            T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(),
            T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD),
        ]),
        "strong": T.Compose([
            T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(),
            T.RandAugment(num_ops=2, magnitude=9),
            T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD),
            T.RandomErasing(p=0.25),
        ]),
    }
    print("Augmentation strategies ready:", list(AUGMENTATIONS.keys()))
else:
    print("[DEMO MODE] Augmentation pipelines are defined with torchvision.transforms:")
    print(" - none      : ToTensor + Normalize only (no randomness)")
    print(" - crop_flip : + RandomCrop(32, padding=4) + RandomHorizontalFlip  (classical CIFAR-10 recipe)")
    print(" - strong    : + RandAugment(2 ops) + RandomErasing(p=0.25)        (stronger / modern recipe)")
    print("EVAL_TRANSFORM (val/test) is ALWAYS just ToTensor + Normalize -- never randomised.")

## Factor 3 — Optimiser: same loss surface, different paths down it

All three optimisers below minimise the same cross-entropy loss on the same weights — what differs is *how* they use the gradient at each step:

- **SGD + momentum** ([Sutskever et al., 2013](https://proceedings.mlr.press/v28/sutskever13.html)) — the classic update, plus a running average ("momentum") of past gradients added to the step, which damps oscillations across steep directions and accelerates movement along shallow, consistent ones. With Nesterov acceleration (used below), the gradient is evaluated at the *look-ahead* point rather than the current one — a small correction that tends to converge slightly faster in practice.
- **Adam** ([Kingma & Ba, 2015](https://arxiv.org/abs/1412.6980)) — keeps a per-parameter running estimate of both the gradient's mean *and* its variance, and rescales each parameter's step by (roughly) the inverse of that variance estimate. This makes Adam largely self-tuning, which is why it usually converges fast with little tuning — but this adaptivity is also often blamed in the literature for **slightly worse generalisation** on image-classification CNNs compared to well-tuned SGD+momentum.
- **AdamW** ([Loshchilov & Hutter, 2019](https://arxiv.org/abs/1711.05101)) — same adaptive mechanism as Adam, but **decouples weight decay from the gradient-based update**. In plain Adam, L2 weight decay is folded into the gradient *before* Adam's adaptive rescaling touches it, so parameters with large gradient-variance estimates get *less* effective decay than intended. AdamW applies decay as a separate, un-rescaled step, closing much of the generalisation gap to SGD.

**Why can't all three share one learning rate?** Adam/AdamW's adaptive rescaling puts their *effective* step size on a totally different scale than SGD's raw gradient step — a rate tuned for SGD (e.g. 0.1) would blow up Adam's training, and a rate suited to Adam (e.g. 1e-3) would make SGD barely move. The code below implements the assignment's **recommended** option (§3.4): a small per-optimiser learning-rate grid, selected using **validation accuracy only** (never test), with the selected rate then frozen for the full-budget run — a legitimate, reportable part of Methods, not "cheating."

The learning-rate **schedule policy** (cosine annealing — [Loshchilov & Hutter, 2017](https://arxiv.org/abs/1608.03983), decaying smoothly along one cosine arc to ~0) is kept identical across optimisers — only the *peak* rate differs. Mixing different schedule shapes *and* different rates would reintroduce exactly the confound OFAT exists to avoid.

→ **Methods**: optimisers, LR search grid and selected rates, weight decay, schedule policy.


In [ ]:
if HAS_TORCH:
    def build_optimizer(name, params, lr, weight_decay=5e-4):
        if name == "sgd_momentum":
            return torch.optim.SGD(params, lr=lr, momentum=0.9, nesterov=True, weight_decay=weight_decay)
        elif name == "adam":
            return torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)
        elif name == "adamw":
            return torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
        raise ValueError(name)

    def build_scheduler(optimizer, n_epochs):
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    # A short per-optimiser LR search (recommended in §3.4): try each candidate
    # for a few epochs, keep the one with the best VALIDATION accuracy, freeze
    # that choice for the full 30-epoch run.
    LR_GRID = {
        "sgd_momentum": [0.2, 0.1, 0.05],
        "adam":         [3e-3, 1e-3, 3e-4],
        "adamw":        [3e-3, 1e-3, 3e-4],
    }
    # Placeholder until the search above is actually run on your machine;
    # replace with the argmax-validation-accuracy rate for each optimiser.
    SELECTED_LR = {k: v[1] for k, v in LR_GRID.items()}
    print("LR search grids (validation-set selected):")
    for k, v in LR_GRID.items():
        print(f"  {k:12s}: {v}")
else:
    SELECTED_LR = {"sgd_momentum": 0.1, "adam": 1e-3, "adamw": 1e-3}
    print("[DEMO MODE] LR search skipped; using the rates a short grid search typically selects:")
    for k, v in SELECTED_LR.items():
        print(f"  {k:12s}: lr = {v}")

## The shared training loop: what must stay identical, made concrete in code

Section 3.1 lists what must be identical in *every* run unless it's the factor under study. In code terms that becomes: one `train_one_epoch`/`evaluate` pair, one loss function (`F.cross_entropy` — the standard choice for single-label multi-class classification: it is the negative log-likelihood of the true class under the model's softmax output, so minimising it directly maximises the probability the model assigns to the correct label), one epoch budget (`N_EPOCHS = 30`), one batch size, and one model-selection rule (best **validation** accuracy). `run_training()` is the single function every one of the 21 runs calls — the only things that change between calls are the `config` dict (architecture/augmentation/optimiser) and the `seed`. Structuring the pipeline this way is what the marking scheme's "Code quality" criterion means by *"the training/evaluation loop are factored so that the three studies share one pipeline"* — and it's what makes the OFAT design actually controlled in practice: if augmentation runs used a subtly different evaluation function than architecture runs, the three studies wouldn't really be comparable.

Note the model-selection discipline inside `run_training`: the checkpoint with the best **validation** accuracy across the 30 epochs is kept, and the **test set is evaluated exactly once**, after training finishes, using that checkpoint. This is the code-level enforcement of "never use the test set for any training or model-selection decision" (§4).

→ **Methods**: loss function, epoch budget, batch size, early-stopping/model-selection rule.


In [ ]:
if HAS_TORCH:
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, Subset

    def train_one_epoch(model, loader, optimizer, device):
        model.train()
        total_loss, correct, n = 0.0, 0, 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * x.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            n += x.size(0)
        return total_loss / n, correct / n

    @torch.no_grad()
    def evaluate(model, loader, device):
        model.eval()
        total_loss, correct, n = 0.0, 0, 0
        all_preds, all_targets = [], []
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            total_loss += loss.item() * x.size(0)
            preds = logits.argmax(1)
            correct += (preds == y).sum().item()
            n += x.size(0)
            all_preds.append(preds.cpu().numpy())
            all_targets.append(y.cpu().numpy())
        y_pred = np.concatenate(all_preds)
        y_true = np.concatenate(all_targets)
        return total_loss / n, correct / n, y_true, y_pred

    def run_training(config, seed, train_set, val_set, test_set, device):
        """One full training run for one (configuration, seed) pair.
        Returns the training curves and the FINAL test-set metrics, computed
        once, using the checkpoint that scored best on the validation set."""
        set_seed(seed)
        model = ARCH_BUILDERS[config["architecture"]](num_classes=N_CLASSES).to(device)
        lr = SELECTED_LR[config["optimizer"]]
        optimizer = build_optimizer(config["optimizer"], model.parameters(), lr)
        scheduler = build_scheduler(optimizer, N_EPOCHS)

        train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
        val_loader   = DataLoader(val_set,   batch_size=256, shuffle=False, num_workers=2)
        test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False, num_workers=2)

        history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
        best_val_acc, best_state = -1.0, None
        for epoch in range(N_EPOCHS):
            tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, device)
            va_loss, va_acc, _, _ = evaluate(model, val_loader, device)
            scheduler.step()
            history["train_loss"].append(tr_loss); history["train_acc"].append(tr_acc)
            history["val_loss"].append(va_loss);   history["val_acc"].append(va_acc)
            if va_acc > best_val_acc:                        # model selection uses VAL only
                best_val_acc = va_acc
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        model.load_state_dict(best_state)
        _, test_acc, y_true, y_pred = evaluate(model, test_loader, device)   # test touched ONCE, at the end
        return {"history": history, "y_true": y_true, "y_pred": y_pred,
                "test_acc": test_acc, "n_params": count_params(model)}

    print("Real training functions defined: train_one_epoch, evaluate, run_training.")
else:
    print("[DEMO MODE] The real training loop (train_one_epoch / evaluate / run_training) is")
    print("defined above exactly as it would run on a GPU. It is not executed in this runtime.")

## Metrics: why accuracy alone is not enough

CIFAR-10 is balanced (equal images per class), so overall **accuracy** is meaningful — but it treats all 10,000 test images as interchangeable, hiding *which* classes a model is good or bad at. `compute_metrics()` below implements the required set (§5):

- **Accuracy** — fraction correct overall.
- **Macro-F1** — compute precision, recall, and F1 *per class*, then average the F1 scores **unweighted** across the 10 classes. Because every class gets equal weight regardless of how easy it is, macro-F1 visibly drops if a model does well on 9 classes but is very poor on one — accuracy alone could mask that.
- **Per-class precision & recall** — precision = "of the images predicted as class X, how many really are X" (false-positive awareness); recall = "of the images that really are X, how many did the model find" (false-negative / miss-rate awareness).
- **Confusion matrix** — the full true-class × predicted-class breakdown, which is what lets you say *which* classes get mistaken for *which* in the Discussion.

**Why does the assignment stress that "micro-averaged F1 is identical to accuracy for a balanced test set"?** Micro-averaging pools every individual prediction together *before* computing precision/recall/F1 — for single-label multi-class problems, total false positives across all classes always exactly equals total false negatives (every wrong prediction is simultaneously a FP for the predicted class and an FN for the true class), which forces micro-precision = micro-recall = micro-F1 = accuracy algebraically. Reporting it as if it were a *different* number from accuracy would be misleading — hence "state clearly which averaging you use."

**Aggregating over seeds, correctly:** `compute_metrics` is called separately for each run's own predictions, and only the resulting numbers (accuracy, macro-F1, …) get averaged across the three seeds afterwards. The assignment explicitly warns against pooling three runs' predictions into one giant set and computing a single metric over that — doing so would produce one point estimate with no notion of run-to-run spread, throwing away exactly the seed-to-seed variability the whole design is built to measure.

The function works with or without scikit-learn — when unavailable, it falls back to a small NumPy implementation of the same precision/recall/F1 definitions computed straight from the confusion matrix, which doubles as a way to see explicitly *what* those metrics compute rather than treating `sklearn.metrics` as a black box.

→ **Methods**: "Describe the evaluation metrics and the averaging used." → **Results**: per-class precision/recall/F1 and confusion matrices for best & worst configurations.


In [ ]:
def compute_metrics(y_true, y_pred, class_names=CLASSES):
    """Accuracy, macro-F1, per-class precision/recall, and the confusion
    matrix. Uses scikit-learn when available; otherwise falls back to a
    pure-NumPy implementation of the same definitions."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    n_classes = len(class_names)
    cm = np.zeros((n_classes, n_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1

    acc = (y_true == y_pred).mean()
    if HAS_SKLEARN:
        macro_f1 = f1_score(y_true, y_pred, average="macro")
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average=None, labels=range(n_classes), zero_division=0)
    else:
        tp = np.diag(cm).astype(float)
        precision = np.divide(tp, cm.sum(0), out=np.zeros(n_classes), where=cm.sum(0) > 0)
        recall    = np.divide(tp, cm.sum(1), out=np.zeros(n_classes), where=cm.sum(1) > 0)
        f1 = np.divide(2 * precision * recall, precision + recall,
                        out=np.zeros(n_classes), where=(precision + recall) > 0)
        macro_f1 = f1.mean()          # unweighted mean over classes -- every class counts equally

    return {"accuracy": acc, "macro_f1": macro_f1,
            "precision": dict(zip(class_names, precision)),
            "recall": dict(zip(class_names, recall)),
            "f1": dict(zip(class_names, f1)),
            "confusion_matrix": cm}

# self-test on a synthetic prediction set: shows this function working
# without needing a trained model or a GPU
rng = np.random.RandomState(0)
demo_true = rng.randint(0, 10, size=2000)
demo_pred = demo_true.copy()
flip_mask = rng.rand(2000) < 0.15          # simulate an ~85%-accurate classifier
demo_pred[flip_mask] = rng.randint(0, 10, size=flip_mask.sum())
m = compute_metrics(demo_true, demo_pred)
print(f"accuracy = {m['accuracy']:.3f}   macro-F1 = {m['macro_f1']:.3f}")
print("per-class recall:", {k: round(v, 2) for k, v in m['recall'].items()})

## The seven configurations, and the self-check that keeps the design honest

This is where the OFAT design from the first section becomes a concrete, checkable data structure. `C1` is the baseline; every other row changes **exactly one** column relative to `C1` (architecture, augmentation, or optimiser — never two at once). The assertion loop below is the programmatic version of the assignment's own instruction (§3.5, "Step 2"): *"Use this table as a self-check on your own design: every row must differ from C1 in exactly one column. If a row differs in two columns, the comparison against C1 no longer isolates a single factor, and the design is broken."*

→ **Methods**: "Describe the baseline configuration and the one-factor-at-a-time design" / "Clearly state what is fixed and what varies in each of the three studies."


In [ ]:
# The 7 configurations from Section 3.5. C1 is the shared baseline; every
# other row changes EXACTLY one column relative to C1.
CONFIGS = [
    {"id": "C1", "architecture": "resnet18",    "augmentation": "crop_flip", "optimizer": "sgd_momentum", "study": "baseline"},
    {"id": "C2", "architecture": "simple_cnn",   "augmentation": "crop_flip", "optimizer": "sgd_momentum", "study": "architecture"},
    {"id": "C3", "architecture": "densenet121",  "augmentation": "crop_flip", "optimizer": "sgd_momentum", "study": "architecture"},
    {"id": "C4", "architecture": "resnet18",     "augmentation": "none",      "optimizer": "sgd_momentum", "study": "augmentation"},
    {"id": "C5", "architecture": "resnet18",     "augmentation": "strong",    "optimizer": "sgd_momentum", "study": "augmentation"},
    {"id": "C6", "architecture": "resnet18",     "augmentation": "crop_flip", "optimizer": "adam",         "study": "optimizer"},
    {"id": "C7", "architecture": "resnet18",     "augmentation": "crop_flip", "optimizer": "adamw",        "study": "optimizer"},
]

baseline = next(c for c in CONFIGS if c["id"] == "C1")
FACTOR_KEYS = ["architecture", "augmentation", "optimizer"]
for c in CONFIGS:
    if c["id"] == "C1":
        continue
    n_diff = sum(c[k] != baseline[k] for k in FACTOR_KEYS)
    assert n_diff == 1, f"{c['id']} differs from baseline in {n_diff} factors, not 1 -- design is broken!"
print(f"Self-check passed: all {len(CONFIGS)-1} non-baseline configs differ from C1 in exactly one factor.")
print(f"{len(CONFIGS)} configurations x {len(SEEDS)} seeds = {len(CONFIGS)*len(SEEDS)} training runs.")

## Running all 21 training runs

`run_all_experiments` is the outer loop that ties the whole Methods pipeline together: for each of the 7 `CONFIGS`, for each of the 3 `SEEDS`, build the right augmented training set, call `run_training(...)`, and collect the returned metrics into one results table plus a dictionary of raw per-run histories/predictions (`raw_runs`) for the plots further down. This is the function you actually call, once, to produce the real "results file" the assignment's deliverables section asks for (`results_df.to_csv("results.csv", index=False)`).

→ **Methods**: this is the concrete implementation of "one shared pipeline" the OFAT design and the marking scheme's "Code quality" criterion both require.


In [ ]:
if HAS_TORCH:
    def run_all_experiments(configs, seeds, device=None):
        device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        rows, raw_runs = [], {}
        train_full = torchvision.datasets.CIFAR10("./data", train=True, download=True)
        targets = np.array(train_full.targets)
        train_idx, val_idx = stratified_split_indices(targets, VAL_FRACTION, seed=1234)  # fixed once, Section 4
        t0 = time.time()
        for config in configs:
            for seed in seeds:
                aug = AUGMENTATIONS[config["augmentation"]]
                train_ds = Subset(torchvision.datasets.CIFAR10("./data", train=True, transform=aug), train_idx)
                val_ds   = Subset(torchvision.datasets.CIFAR10("./data", train=True, transform=EVAL_TRANSFORM), val_idx)
                test_ds  = torchvision.datasets.CIFAR10("./data", train=False, transform=EVAL_TRANSFORM)
                result = run_training(config, seed, train_ds, val_ds, test_ds, device)
                metrics = compute_metrics(result["y_true"], result["y_pred"])
                rows.append({"config_id": config["id"], "seed": seed, "architecture": config["architecture"],
                             "augmentation": config["augmentation"], "optimizer": config["optimizer"],
                             "study": config["study"], "n_params_M": result["n_params"] / 1e6,
                             "test_accuracy": metrics["accuracy"] * 100, "macro_f1": metrics["macro_f1"]})
                raw_runs[(config["id"], seed)] = {**result, "confusion_matrix": metrics["confusion_matrix"]}
                print(f"{config['id']} seed={seed}: acc={metrics['accuracy']*100:.2f}%  "
                      f"macro-F1={metrics['macro_f1']:.3f}  ({time.time()-t0:.0f}s elapsed)")
        return pd.DataFrame(rows), raw_runs

    # For a real run:
    # results_df, raw_runs = run_all_experiments(CONFIGS, SEEDS)
    # results_df.to_csv("results.csv", index=False)
    print("run_all_experiments defined. Call it on a machine with a GPU to produce real results.csv.")
else:
    print("[DEMO MODE] run_all_experiments is defined above exactly as it would run for real; "
          "not executed here. The next cell fills results_df with illustrative placeholder numbers instead.")

## From here on: Results

Everything above this point (imports/seeding, splits, architectures, augmentations, optimisers, the training loop, the metrics function, and the `CONFIGS` table) is **Methods** material — it describes *how* the experiment is built, and every sentence in your paper's Methods section should be traceable to one of those cells.

Everything below is **Results**: it consumes the *output* of running that pipeline 21 times — one row per (configuration, seed) pair, with the final test accuracy, macro-F1, and predictions for each run — and turns it into the tables, curves, and confusion matrices §8 asks for.

**About the numbers below.** As explained at the top of this notebook, no training actually ran here (no GPU/PyTorch in this environment, and 21 real runs would take hours). The table below is filled in with **illustrative, literature-informed placeholder numbers** — plausible for a from-scratch, ~30-epoch, 32×32 CIFAR-10 budget, not benchmark/state-of-the-art figures (those use far more epochs, larger models, or pretraining, which this assignment explicitly forbids). Their only job is to let the aggregation, plotting, and Discussion code below run against *something* so you can see the full pipeline end-to-end and reuse the code unchanged once you plug in real numbers from `run_training`.


In [ ]:
# Illustrative per-run results: 7 configurations x 3 seeds = 21 rows.
# Replace this hard-coded list with the real output of
#     results_df, raw_runs = run_all_experiments(CONFIGS, SEEDS)
# (a thin loop that calls run_training(config, seed, ...) for every
# configuration/seed pair and collects the returned metrics -- see the
# outline in the markdown cell above `run_training` for what that loop
# looks like; it is intentionally omitted here since it needs a GPU to run).
ILLUSTRATIVE_ACC = {   # (test accuracy % per seed 0,1,2)
    "C1": [89.4, 88.9, 89.3],   # ResNet-18, crop+flip, SGD+momentum (baseline)
    "C2": [78.1, 77.0, 77.7],   # Simple CNN
    "C3": [90.7, 89.6, 90.0],   # DenseNet-121
    "C4": [86.0, 84.5, 85.4],   # ResNet-18, no augmentation
    "C5": [90.3, 88.8, 89.7],   # ResNet-18, strong augmentation
    "C6": [88.5, 87.3, 87.9],   # ResNet-18, crop+flip, Adam
    "C7": [89.1, 88.3, 88.7],   # ResNet-18, crop+flip, AdamW
}
ILLUSTRATIVE_F1 = {
    "C1": [0.892, 0.887, 0.891], "C2": [0.779, 0.768, 0.775], "C3": [0.906, 0.895, 0.899],
    "C4": [0.858, 0.843, 0.852], "C5": [0.901, 0.886, 0.895], "C6": [0.883, 0.871, 0.877],
    "C7": [0.889, 0.881, 0.885],
}
PARAMS_M = {"C1": 11.17, "C2": 0.59, "C3": 6.96, "C4": 11.17, "C5": 11.17, "C6": 11.17, "C7": 11.17}
SEC_PER_EPOCH = {"C1": 22, "C2": 8, "C3": 55, "C4": 20, "C5": 27, "C6": 22, "C7": 22}

cfg_by_id = {c["id"]: c for c in CONFIGS}
rows = []
for cid, accs in ILLUSTRATIVE_ACC.items():
    for seed, acc, f1 in zip(SEEDS, accs, ILLUSTRATIVE_F1[cid]):
        c = cfg_by_id[cid]
        rows.append({"config_id": cid, "seed": seed, "architecture": c["architecture"],
                     "augmentation": c["augmentation"], "optimizer": c["optimizer"], "study": c["study"],
                     "n_params_M": PARAMS_M[cid], "sec_per_epoch": SEC_PER_EPOCH[cid],
                     "test_accuracy": acc, "macro_f1": f1})
results_df = pd.DataFrame(rows)
results_df.head(9)

# One row per training run -- this DataFrame is exactly the shape of the
# "results file" (CSV/JSON) the assignment's deliverables ask for.

## Aggregating over seeds: the three tables Results needs

`paper_table()` below does exactly what §5 ("Aggregating over seeds") and §8 ("Results — one table per factor study, as mean ± standard deviation over the three seeds") require: group the per-run rows by configuration, average the **already-computed per-run metrics**, and report the standard deviation alongside the mean — never pooling raw predictions across seeds first (see the Metrics section above for why that would be wrong).

`C1` deliberately reappears in all three tables with exactly the same numbers — that repetition is intended (§3.5, Step 4): it is the fixed reference point every study is measured against, not a duplication error.


In [ ]:
def paper_table(df, study_name, baseline_id="C1"):
    """One row per configuration in a study, mean +/- std over the 3 seeds --
    the table format required in Results for each factor."""
    sub = df[(df["study"] == study_name) | (df["config_id"] == baseline_id)].copy()
    sub = sub.drop_duplicates(subset=["config_id", "seed"])
    agg = sub.groupby("config_id").agg(
        architecture=("architecture", "first"), augmentation=("augmentation", "first"),
        optimizer=("optimizer", "first"), n_params_M=("n_params_M", "first"),
        acc_mean=("test_accuracy", "mean"), acc_std=("test_accuracy", "std"),
        f1_mean=("macro_f1", "mean"), f1_std=("macro_f1", "std"),
    ).round(3)
    return agg

arch_table = paper_table(results_df, "architecture")
aug_table  = paper_table(results_df, "augmentation")
opt_table  = paper_table(results_df, "optimizer")

print("=== Study 1: Architecture ===")
print(arch_table[["architecture", "n_params_M", "acc_mean", "acc_std", "f1_mean", "f1_std"]])
print("\n=== Study 2: Augmentation ===")
print(aug_table[["augmentation", "acc_mean", "acc_std", "f1_mean", "f1_std"]])
print("\n=== Study 3: Optimizer ===")
print(opt_table[["optimizer", "acc_mean", "acc_std", "f1_mean", "f1_std"]])

# These three printed tables ARE Table 1 / Table 2 / Table 3 of your Results
# section (formatted as a proper LaTeX/IEEE table when you write the paper).

## Training curves: the evidence behind "convergence speed" and "overfitting"

A single final-accuracy number cannot show *how* a run got there. Logging train/validation loss and accuracy every epoch (required by §5, "Training curves") is what lets you say things like "Adam reached 85% validation accuracy in half as many epochs as SGD" or "removing augmentation widens the train/val gap" — statements a results table alone cannot support.

The plotting code below is written against `raw_runs[(config_id, seed)]["history"]`, a dict with `train_acc`/`val_acc`/`train_loss`/`val_loss` arrays of length `N_EPOCHS` — exactly what `run_training()` returns per run. Since no real training ran in this environment, `raw_runs` isn't populated here; **this cell shows the real plotting code you'll use once `raw_runs` exists** (uncomment the calls to run it against your real results).

→ **Results**: "Training curves for the runs you discuss."


In [ ]:
def plot_training_curves(raw_runs, pairs, title):
    """pairs: list of (config_id, seed, label, color). Plots train (dashed)
    vs val (solid) accuracy for each entry on one axis."""
    fig, ax = plt.subplots(figsize=(6, 4))
    for cid, seed, label, color in pairs:
        hist = raw_runs[(cid, seed)]["history"]
        epochs = range(1, len(hist["val_acc"]) + 1)
        ax.plot(epochs, hist["train_acc"], color=color, linestyle="--", alpha=0.6)
        ax.plot(epochs, hist["val_acc"], color=color, label=label)
    ax.set_xlabel("epoch"); ax.set_ylabel("accuracy (%)"); ax.set_title(title)
    ax.legend(fontsize=8)
    plt.tight_layout()
    return fig

# Example usage once `raw_runs` holds real training histories:
#
# plot_training_curves(raw_runs, [
#     ("C1", 0, "C1 baseline (crop+flip)", "tab:blue"),
#     ("C4", 0, "C4 no augmentation",      "tab:red"),
#     ("C5", 0, "C5 strong augmentation",  "tab:green"),
# ], title="Augmentation study: train (dashed) vs val (solid)")
#
# plot_training_curves(raw_runs, [
#     ("C1", 0, "SGD+momentum", "tab:blue"),
#     ("C6", 0, "Adam",         "tab:orange"),
#     ("C7", 0, "AdamW",        "tab:purple"),
# ], title="Optimizer study: convergence speed")
print("plot_training_curves defined -- call it once raw_runs is populated by a real run.")

## Confusion matrices and error analysis

`raw_runs[(config_id, seed)]["y_true"]`/`["y_pred"]` (returned by `run_training`, via `evaluate`) feed straight into `compute_metrics()`'s confusion matrix. Row-normalising it (dividing each row by that true class's total count) turns raw counts into "given the true class, what fraction of the time did the model predict each class" — much easier to compare across classes with different error rates at a glance.

§5 requires this **for at least your best and worst configurations** (by mean accuracy), which is exactly what `best_id`/`worst_id` below select automatically from `results_df` rather than being hand-picked.

→ **Results**: "Confusion matrices and per-class metrics for at least the best and worst configurations."


In [ ]:
def plot_confusion_matrix(cm, class_names, title, ax):
    cm_norm = cm.astype(float) / cm.sum(1, keepdims=True)
    ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(class_names))); ax.set_xticklabels(class_names, rotation=90, fontsize=7)
    ax.set_yticks(range(len(class_names))); ax.set_yticklabels(class_names, fontsize=7)
    ax.set_title(title); ax.set_xlabel("predicted"); ax.set_ylabel("true")

best_id  = results_df.groupby("config_id")["test_accuracy"].mean().idxmax()
worst_id = results_df.groupby("config_id")["test_accuracy"].mean().idxmin()
print(f"Best mean accuracy : {best_id}   Worst mean accuracy : {worst_id}")

# Example usage once raw_runs holds real confusion matrices:
#
# fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
# plot_confusion_matrix(raw_runs[(best_id, 0)]["confusion_matrix"],  CLASSES, f"{best_id} (best, seed 0)",  axes[0])
# plot_confusion_matrix(raw_runs[(worst_id, 0)]["confusion_matrix"], CLASSES, f"{worst_id} (worst, seed 0)", axes[1])
# plt.tight_layout()
print("plot_confusion_matrix defined -- call it once raw_runs is populated by a real run.")

## Effect size vs. seed noise: turning tables into a ranking

This is the single most important plot in the whole paper: for each non-baseline configuration, the bar shows its accuracy **difference from the `C1` baseline** (`delta_vs_baseline`), with an error bar equal to that configuration's own seed-to-seed standard deviation, against a shaded band showing the *baseline's* seed-to-seed noise. A bar whose error bar crosses zero, or overlaps the shaded band, is a difference **within noise** — the assignment is explicit that reporting this honestly ("say so") is worth more marks than claiming an unsupported winner.

→ **Results → Discussion bridge**: this chart is the evidence for "which of the three factors has the largest effect" and "is that effect larger than the variation between your three seeds" (§3.7).


In [ ]:
baseline_mean = results_df[results_df.config_id == "C1"]["test_accuracy"].mean()
baseline_std  = results_df[results_df.config_id == "C1"]["test_accuracy"].std()

summary = results_df.groupby("config_id").agg(
    study=("study", "first"), mean_acc=("test_accuracy", "mean"), std_acc=("test_accuracy", "std"),
).drop("C1")
summary["delta_vs_baseline"] = summary["mean_acc"] - baseline_mean

fig, ax = plt.subplots(figsize=(8, 4.5))
colors = {"architecture": "tab:blue", "augmentation": "tab:green", "optimizer": "tab:orange"}
ax.bar(summary.index, summary["delta_vs_baseline"],
       yerr=summary["std_acc"], color=[colors[s] for s in summary["study"]], capsize=4)
ax.axhline(0, color="k", linewidth=0.8)
ax.axhspan(-baseline_std, baseline_std, color="grey", alpha=0.2, label="baseline seed-to-seed noise band")
ax.set_ylabel("delta accuracy vs C1 baseline (pp)")
ax.set_title("Effect size of each configuration relative to the baseline's own noise")
ax.legend(fontsize=8)
plt.tight_layout()

print(summary[["study", "mean_acc", "std_acc", "delta_vs_baseline"]].round(2))

## Discussion & Conclusion — worked example, using the illustrative numbers above

This is the section the assignment weights most heavily in the paper (25% "Results and discussion", and it's where "Reaching the top band" lives — arguing from effect size, analysing failure modes, discussing limitations). Below, the assignment's own §3.7 "Questions to consider" are answered **using the illustrative numbers produced above**, as a worked example of the *reasoning pattern* — not as a claim about what your real run will show. When you replace the placeholders with real training results, redo this reasoning against your actual numbers; the pattern stays the same.

**Which factor has the largest effect?** In the bar chart above, `C2` (Simple CNN) sits far outside the grey baseline-noise band — architecture, specifically "plain CNN vs. a modern backbone," is the dominant factor. `C3` (DenseNet-121) is only marginally above `C1` and largely inside the noise band — more capacity / a different backbone family than ResNet-18 doesn't clearly help further at this epoch budget. Augmentation's clearest effect is the *negative* direction: removing it (`C4`) costs several points and is well outside the noise band. Optimiser produces the smallest and noisiest effect of the three: `C6` (Adam) sits below the baseline but close to its noise band, and `C7` (AdamW) is essentially tied with the SGD+momentum baseline. → **Ranking (this worked example): architecture > augmentation > optimiser**, and only architecture and augmentation clear the seed-to-seed noise floor with real confidence.

**Is the effect larger than seed variation?** This is the entire point of plotting `delta_vs_baseline` next to `yerr=std_acc`. A bar (with its error bar) that does not cross zero, and does not overlap the baseline's own shaded noise band, supports a claim; one that does overlap does not — regardless of which configuration happens to have the higher point estimate. `C3` vs `C1` in this worked example is the textbook case of "looks like a win, isn't a defensible one": roughly a 1-point gap sitting inside a noise band of comparable size.

**Does augmentation help all architectures equally?** The OFAT design as specified only tests augmentation *on the baseline architecture* (ResNet-18) — a real limitation, not an oversight. §3.6's optional interaction study exists precisely because OFAT cannot answer "does augmentation help the Simple CNN by the same amount as ResNet-18?" If your group has compute budget left, this is the single highest-value optional addition (it's explicitly called out in "Reaching the top band").

**Optimiser: accuracy, convergence speed, or both?** Once real training curves exist (from `plot_training_curves`), look at their shape, not just the endpoint: even where final accuracy ends up close, Adam-family optimisers typically reach a given validation accuracy in fewer epochs than SGD+momentum, which starts slower but often catches up in final accuracy. The honest answer is often "both, in different ways" — optimiser choice can change *how fast* you get somewhere without changing *how good* the final answer is, a distinction only visible because training curves were logged every epoch, not just the final number.

**Accuracy per parameter / per training time?** Exactly what the `n_params_M` and `sec_per_epoch` columns in `results_df` are for. In this worked example, `C2` (Simple CNN, 0.6 M params, ~8 s/epoch) reaches ~78% — worse absolute accuracy, but at a small fraction of ResNet-18's parameter count and epoch time; `C3` (DenseNet-121) needs ~2.5× ResNet-18's epoch time for a gain that doesn't clear the noise floor. That accuracy-per-cost trade-off is exactly the comparison "Reaching the top band" asks for.

**Do configurations make the same kind of mistakes?** Once real confusion matrices exist (from `plot_confusion_matrix`) for the best and worst configurations, compare their off-diagonal hot-spots. In CIFAR-10 the classic confusions are cat↔dog and automobile↔truck, because those pairs share more low-level visual structure (fur texture and body shape; both are wheeled vehicles seen from similar angles) than they share with the other eight classes. If a weaker configuration's errors are spread much more broadly across *all* classes rather than concentrated on those familiar pairs, that's itself informative — it suggests the weaker model hasn't learned the same *kind* of features as the stronger ones, not just fewer of them.

**Which classes are hardest, and is that consistent?** Read recall per class off `compute_metrics()`'s output for each run. In CIFAR-10, "cat" is close to universally the hardest class in the published literature (lowest recall almost regardless of architecture) — if your own confusion matrices agree across configurations, that's a finding about the *dataset* (cats have huge intra-class visual variation: pose, colour, fur pattern), not about any one model, and is worth saying explicitly.

**Does the train/val gap behave as expected as augmentation strengthens?** This is exactly what the augmentation-study training-curve panel is for: `C4` (no augmentation) should show the widest train/val gap, `C1` (crop+flip) a smaller one, and `C5` (strong augmentation) the smallest — sometimes even with training accuracy *below* that of the weaker-augmented runs, because strong augmentation makes the training task itself harder, not just more general. If your real run shows `C5`'s gap *not* shrinking, that's a legitimate, interesting finding (worth discussing rather than hiding) — it could mean the augmentation needs more than 30 epochs to pay off, exactly the kind of "fixed epoch budget" limitation §8 asks you to discuss explicitly.

**Limitations to state explicitly (§8 requires this list):**
- **Fixed epoch budget** — a short, identical budget makes runs affordable to repeat 21 times, but likely under-trains the stronger/slower-converging configurations (e.g. `strong` augmentation, `DenseNet-121`) relative to what they could reach with a longer schedule.
- **Each factor measured at only one baseline setting of the other two** — pure OFAT, not a full factorial; architecture's effect is only characterised *at* `crop_flip` augmentation and SGD+momentum, and may not generalise elsewhere (exactly what §3.6's optional interaction study exists to probe).
- **Only 3 seeds** — enough to report a standard deviation, not enough to fit a serious distribution; a spread estimated from n=3 is itself noisy.
- **32×32 resolution** — much lower detail than the 224×224 inputs these architectures were originally designed around; conclusions about "which architecture is best" may not transfer to higher-resolution problems.
- **Many comparisons against one test set** — with 21 runs (and derived comparisons across all of them), the single best number among them is expected to look good partly by chance (§3, "Comparing fairly"); don't let the leaderboard-topping run stand in for "the winning configuration" without the seed-averaged table backing it up.

→ **This entire subsection is Discussion & Conclusion material** — the "answer your research questions and state whether each hypothesis was supported," "rank the three factors by effect size," and "discuss limitations" bullets in §8 map directly onto the points above.


## Notebook section → paper section, at a glance

| Notebook section | Feeds paper section | What it proves |
|---|---|---|
| Env/capability flags, `set_seed` | Methods — seeds & hardware | reproducibility |
| Constants (classes, split, epochs, batch) | Methods — shared baseline, what's fixed | OFAT control list (§3.1) |
| `stratified_split_indices` | Methods — train/val/test split | fixed, stratified, reused split |
| Model definitions + parameter counts | Methods — architectures; Related Work justification | 32×32 adaptation, capacity |
| Augmentation transforms | Methods — augmentation strategies | train-only, progression |
| Optimiser/LR grid | Methods — optimisers, LR choice | fair comparison policy |
| `train_one_epoch` / `evaluate` / `run_training` | Methods — loss, epoch budget, model selection | no test-set leakage |
| `compute_metrics` | Methods — evaluation metrics & averaging | accuracy / macro-F1 / per-class |
| `CONFIGS` + self-check | Methods — OFAT design table | "exactly one factor differs" |
| `run_all_experiments` → `results_df` | Results — results file | one row per run |
| `paper_table` (3 tables) | Results — Table 1 / 2 / 3 | mean ± std per factor |
| `plot_training_curves` | Results — training curves | convergence / overfitting evidence |
| `plot_confusion_matrix` | Results — best/worst error analysis | which classes get confused |
| effect-size bar chart | Results → Discussion bridge | effect size vs. noise |
| "Discussion & Conclusion" walkthrough | Discussion & Conclusion | answers to §3.7 questions |


## Turning this into your real submission

1. Install `torch`, `torchvision` (and optionally `scikit-learn`) in an environment with a GPU; `HAS_TORCH`/`HAS_SKLEARN` flip to `True` and every guarded cell above switches from its printed placeholder message to the real path automatically.
2. Let `torchvision.datasets.CIFAR10(..., download=True)` fetch the real data (needs internet on that machine).
3. Do your own literature-review pass to justify (or replace) the three architectures / augmentations / optimisers in `CONFIGS` and `LR_GRID` — copying this notebook's choices without your own Related Work reasoning is exactly the "simply reproducing an existing notebook" the assignment says is not sufficient to pass.
4. Run `results_df, raw_runs = run_all_experiments(CONFIGS, SEEDS)` for real, then `results_df.to_csv("results.csv", index=False)` — expect this to take a while; that's why the epoch budget must stay modest and identical across configs (§3.1).
5. Delete the "illustrative results" hard-coded cell and re-run `paper_table`, `plot_training_curves`, `plot_confusion_matrix`, and the effect-size chart against your real `results_df`/`raw_runs` — none of that code needs to change, it was written to consume exactly that shape of data.

Sections 6 (group work), 7 (deliverables/ZIP packaging), 9 (resource links) and 10 (marking scheme) aren't covered here, per your request — but two things are cheap to get right early anyway: the notebook and paper both need every group member's **student ID** inside them (not names), and §8's Abstract/Introduction/Related Work still need writing even though this notebook's explanations focus on Methods/Results/Discussion.


---

**One aside on the source PDF.** The last page of `main.pdf`, after the "— End of the document —" line, has two short handwritten-looking notes ("Secmentation class fot the last exam", "write about your exam and what you have found…"). They read like personal margin notes rather than part of the assignment text (unnumbered, not in the document's formatting, not matching anything in the marking scheme), so they were treated as unrelated to this assignment and ignored when building this notebook — flagging them here in case they were meant for something else and ended up in this file by mistake.
